In [ ]:
## In this tutorial, we will open an existing System file and navigate the hierarchy of classes inside.

In [ ]:
import os
from scope.read_write import load_binary

In [ ]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data"
data_folder = os.path.abspath('.')+'/Data/'
## Loads the System object from a binary file, provided in the tutorial folder
sys = load_binary(f"{data_folder}ABITEM.npy")
#sys = load_binary("/Users/sergivela/Documents/SCOPE/Database_SCO/Test_V1/Systems/ABITEM/ABITEM.npy")

In [ ]:
## All objects in SCOPE have a __repr__ method, so printing shows a summary of the object
print(sys)

In [ ]:
#The System Object mainly stores paths and sources. 

# A source is a chemical entity. There are two main types of sources: CELLs and SPECIEs:  
# -The difference between CELLs and SPECIEs is that the former is a periodic structure, so it must include cell parameters.  
# -A SPECIE is a class with various subclasses: Molecule, Ligand, and Group. Ligands and Groups only appear in Transition Metal Complexes.

# In this specific example, the system ABITEM is associated with 6 sources:
    #0: cell ABITEM01 H104-C92-N24-O4-S8-Fe4 
    #1: cell ABITEM H104-C92-N24-O4-S8-Fe4 
    #2: cell ref_hs_cell H104-C92-N24-O4-S8-Fe4 
    #3: cell ref_ls_cell H104-C92-N24-O4-S8-Fe4 
    #4: specie ref_hs_mol H18-C20-N6-S2-Fe 
    #5: specie ref_ls_mol H18-C20-N6-S2-Fe 

    # Sources 0-1 are the original cell objects imported from cell2mol
    # Sources 2-3 are the cell objects identified as reference HS and LS unit cells. This will undergo the Solid-State computations in tutorial XXX
    # Sources 4-5 are the specie objects identified as reference HS and LS molecules. This will undergo the Isolated computations in tutorial XXX

In [ ]:
# We will navigate both CELL and SPECIE Objects below.

## Sources 1: The CELL Class


In [ ]:
#All objects in SCOPE can be printed, which displays their key information. 
cell = sys.sources[0]
print(cell)

In [ ]:
# Have a look at the CELL object above.

# -It has 236 atoms, whose coordinates are stored as cell.coord, and labels as cell.labels.
print("Labels and Coordinates:", cell.labels[0], cell.coord[0]) 
# -It has cell parameters, stored as cell.cell_param
print("Cell Parameters:", cell.cell_param)
# -It has 8 Molecules, stored inside cell.moleclist
print("Number of Molecules:", len(cell.moleclist))
# -It has 2 types of molecules, with formulae:
for mol in cell.refmoleclist:
    print("Unique Molecule Formula:", mol.formula)

In [ ]:
# You can also run CELL-class functions, like cell.check_fragmentation(), which checks if any molecule appears fragmented due to how the cell is cut
cell.check_fragmentation()

In [ ]:
# This CELL is, indeed, a specific CELL subclass created by the Spin_Crossover add-on. Notice the subtype
print(cell.subtype)
# This type of CELL is created from cell2mol data, and is associated with a .cif-class object, whose information you can also visualize here
print(cell.cif)

In [ ]:
# You can also view the unit cell with cell.view(), which opens a 3D viewer
cell.view(size='large')

## Sources 2: The SPECIE Class

In [ ]:
# Hopefully, you can identify the 4 transition metal complexes (TMCs) and the 4 solvent molecules in the viewer above.
# In any case, you can access those molecules through the cell.moleclist attribute. 

# MOLECULE-class objects are a subclass of SPECIE, which more broadly refers to any chemical specie, including ligands in transition metal complexes

### 2.1) Molecule Sub-Class

In [ ]:
# Hopefully, you can identify the 4 transition metal complexes (TMCs) and the 4 solvent molecules in the viewer above.
# In any case, you can access those molecules through the cell.moleclist attribute. Taking the first molecule as an example:
mol = cell.moleclist[0]
# Again, the key information is displayed
print(mol)


In [ ]:
# Notice the molecule is of type=SPECIE, and subtype=MOLECULE. As mentioned above, MOLECULE is a subclass of SPECIE. 
# Also, this molecule is a TMC, as shown by:
print(mol.iscomplex)
# As a result, it has a ligand list (mol.ligands), which contains 3 Ligand objects:
print(len(mol.ligands), "ligands in this TMC")
# And a list of metal atoms:
print(len(mol.metals), "metal atom(s) in this TMC")

In [ ]:
## Apart from attributes, species have few useful functions, e.g.:
print(mol.get_centroid())

In [ ]:
## All species can be visualized with the .view() method:
mol.view()

### 2.2) Ligand Sub-Class

In [ ]:
## You can access the ligand objects associated with the molecule
lig = mol.ligands[0]
print(lig)

In [ ]:
## The LIGAND is also a subclass of SPECIE (type=specie, subtype=ligand). It has some useful functions too, e.g.:
print(lig.get_denticity())
## This one retreives the atom object of all atoms that are connected to a metal, in this case only an N atom:
print(lig.get_connected_metals())

In [ ]:
## The cell objects in this tutorial were imported from cell2mol CELL objects. 
## Ligands in these cells are associated with an rdkit object...
lig.rdkit_obj  

In [ ]:
## ... and a Smiles
lig.smiles

In [ ]:
## In transition metal complexes, molecule smiles are the ligand smiles concatenated. New better ways have been proposed, but not yet implemented in SCOPE.
mol.smiles

### 2.3) Metal Sub-Class

In [ ]:
## METAL is a subclass of ATOM, which we will see immediately. 
## You can access the METAL object from the ligand with (1) lig.get_connected_metals(), as we've seen before, or (2) From the molecule: 
met = mol.metals[0]
print(met)

In [ ]:
## Again, METAL is a subclass of ATOM (type=atom, subtype=metal). Apart from attributes, it also has useful functions, e.g.:
met.get_coord_sphere()          ## These are the atoms that are coordinated to the metal
met.get_coord_sphere_formula()  ## This is its formula

In [ ]:
#met.get_cshm()   ## This one retrieves the Coordination Shape Measure of the metal.
                  ## The CShM quantifies how close the coordination environment is to an ideal geometry, which is an octahedron by default.
                  ## Unfortunately, the package that computes it (Cosymlib) introduces dependency issues during the installation of SCOPE 
                  ## If you're skilled enough, you can try to install it manually

### 2.4) Going Upwards: Parents

In [ ]:
## So far, we went down in the hierarchy: System -> Cell -> Molecule -> Metal/Ligand. You can also go back up:

# This retrieves the parent CELL object of the ligand
parent = lig.get_parent('cell') 
print(parent)

In [ ]:
# And this retrieves the index of the ligand atoms in the parent molecule, so you can easily locate the child's atoms in parent attributes
idx = lig.get_parent_indices('molecule') 
print("Ligand Atom Indices in Parent's Molecule class:", idx)

for i, pi in enumerate(idx):
    lig_atom = lig.atoms[i]                                             ## i is the index in ligand atoms: 0,1,2 
    mol_atom = lig.get_parent('molecule').atoms[pi]                     ## pi is the index of the same atom in the parent molecule's atoms: 2,7,41
    print(i, pi, lig_atom.label, mol_atom.label, lig_atom == mol_atom)  ## Here, it compares both atoms. If TRUE, they are the same

### The ATOM Class

In [ ]:
# As we've briefly seen above, species have atoms. 
len(mol.atoms)

In [ ]:
## The first atom of the molecule happens to be the metal. The second is a regular atom. Notice the different subtype
for at in mol.atoms[0:10]:                                  ## Printing the first 10 atoms
    print(at.get_parent_index('molecule'), at.label, at.subtype, at.charge, at.madjnum, at.adjnum)

## at.charge stores the formal atomic charge.
## at.madjnum is the number of adjacencies the atom has with a metal center.
## at.adjnum is the total number of adjacencies the atom has with any other atom

### Charge and Spin

In [ ]:
## Because cell2mol infers charge, this info is also stored in the objects:
for mol in cell.moleclist:
    print(f"Formula: {mol.formula}, Charge: {mol.charge}, Spin: {mol.spin}")

In [ ]:
## And also for the ligands and metals inside the molecules:
mol = cell.moleclist[0]
for lig in mol.ligands:
    print(f"Ligand Formula: {lig.formula}, Charge: {lig.charge}, Spin: {lig.spin}")
for met in mol.metals:
    print(f"Metal: {met.label}, Charge: {met.charge}, Spin: {met.spin}")

In [ ]:
## All the way down to the ATOM objects, which are the actual bearers of spin and charge:
for at in mol.atoms:
    print(f"Atom: {at.label}, Charge: {at.charge}, Spin: {at.spin}")

In [ ]:
## IMPORTANT: The charge and spin attribute for the SPECIE (molecules, ligands) and CELL classes are computed as the sum of the charge and spin of their constituent ATOM objects.mol
## - By "spin" we mean the total number of unpaired electrons, i.e. a spin of 1 means one unpaired electron (doublet), a spin of 2 means two unpaired electrons (triplet), etc.
## - If you want to modify their charge, you should modify the charge/spin of an atom object directly: eg: at.set_charge(1), at.set_spin(2), where "at" is an ATOM object.
##   Alternatively, you can use spec.set_atomic_charges([list_of_charges]) and spec.set_atomic_spins([list_of_spins]) to set all atomic charges/spins of a SPECIE (spec) at once.

## Here we take atom 0, which should be an Fe atom,
at = mol.atoms[0]

## We set it as a high-spin center, with 4 unpaired electrons (spin=4)
at.set_spin(4)

## When introducing the spin, the multiplicity and ms values are also stored as attributes:
print(f"Spin: {at.spin}, Ms: {at.ms}, Multiplicity: {at.multiplicity}")

## When Working with the STATE class in tutorial number 2, we will see how we can rapidly make changes to the charge and spin of specific atoms (typically metals) in the state. 

### Bond Class

In [ ]:
## Bonds are also stored as objects, with very limited functionality for now:
print(at.bonds)

In [ ]:
## The formal bond order (bond.order) value above is extracted from the rdkit object, if available